# SR v2 structural research notebook

This notebook is exploratory and calibration-free. It replays independent source timeframes and displays structural lifecycle evidence only.

In [ ]:
import atexit

from IPython.display import IFrame, display

from libs.models.sr_v2.config import SRV2ConfigResolver
from libs.models.sr_v2.research_lab import (
    SRV2ResearchNotebookConfigResolver,
    candidate_table,
    compose_research,
    feature_table,
)

In [ ]:
model_config = SRV2ConfigResolver.from_yaml('configs/sr_v2.yaml').resolve()
research_config = SRV2ResearchNotebookConfigResolver.from_yaml('configs/sr_v2_research_notebook.yaml').resolve()
ALLOW_PROVIDER_FETCH = False
provider_adapter = None

## Explicit provider opt-in

The default path is cache-only and makes zero provider calls. The next cell is the only provider construction site; it remains disabled unless this notebook toggle and the research YAML source mode are explicitly changed.

In [ ]:
if ALLOW_PROVIDER_FETCH:
    from libs.market_data import BinanceNativeAdapter

    provider_adapter = BinanceNativeAdapter()

In [ ]:
composition = None
try:
    composition = await compose_research(
        model_config,
        research_config,
        adapter=provider_adapter,
        allow_provider_fetch=ALLOW_PROVIDER_FETCH,
    )
except FileNotFoundError:
    print('No verified local cache is available; cache-only replay is not started.')
if composition is not None:
    trace = composition.trace
    source_manifest = composition.source_manifest
    atexit.register(composition.close)

In [ ]:
if composition is not None:
    features = feature_table(trace)
    candidates = candidate_table(trace)
    lifecycle_intervals = trace.lifecycle_intervals
    touch_episodes = trace.touch_episodes
    display({
        'research_status': 'RESEARCH_ONLY',
        'identity_mode': trace.identity_mode,
        'lineage_disclosure': 'WINDOW_RELATIVE disables exact lifetime and pre-boundary predecessor claims',
        'structural_only': True,
        'source_provenance': [
            {
                'timeframe': entry['timeframe'],
                'source_mode': entry['source_mode'],
                'cache_access_mode': entry['cache_access_mode'],
                'origin_mode': entry['origin_mode'],
                'acquisition_cutoff': entry['acquisition_cutoff'],
                'acquisition_evidence_sha256': entry['acquisition_evidence_sha256'],
            }
            for entry in source_manifest.entries
        ],
    })
    display(features[:10], candidates[:10], lifecycle_intervals[:10], touch_episodes[:10])

In [ ]:
if composition is not None:
    for timeframe, url in composition.urls:
        display(
            IFrame(
                src=url,
                width='100%',
                height=research_config.display['iframe_height'],
                title=f'SR v2 {timeframe} structural research',
            )
        )

## Manual cleanup (run after inspecting the IFrames)

The server remains available for inspection after Run All. Cleanup is bounded to this session's owned workspace and is also registered for kernel exit.

In [ ]:
CLOSE_VIEWER_NOW = False
if CLOSE_VIEWER_NOW and composition is not None:
    composition.close()
    composition.close()